Predicting Interaction Outcomes in Digital Commerce Systems

In [2]:
#Importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve
from imblearn.over_sampling import SMOTE

from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix
from scipy.stats import chi2_contingency, normaltest

ModuleNotFoundError: No module named 'pandas'

In [1]:
#Import dataset
Data = pd.read_csv("Code Storm Dataset.csv")

NameError: name 'pd' is not defined

In [ ]:
#To check null values,data type & rows/columns
print(Data.info())

In [ ]:
print(Data.isnull().sum())

In [ ]:
#Sanity‑checking dataset at the start
Data.head()

#Question-1>Perform a detailed exploratory analysis of the dataset

In [ ]:
#Pattern/anomalies analysis
Data['avg_rating'].plot(kind="hist", bins=10, title="Distribution of Ratings")
plt.xlabel("Rating"); plt.ylabel("Count"); plt.show()

In [ ]:
#Customer engagement across product categories
grouped_data = Data.groupby('product_category')['num_reviews'].sum()
grouped_data.plot(kind="bar", color="red", title="Total Reviews by Category")
plt.ylabel("Total Reviews"); plt.show()

In [ ]:
#Summary of statistics
Data.describe()

In [ ]:
#To check outliers
sns.boxplot(x="brand_tier", y="avg_rating", data=Data)
plt.title("Brand Tier vs Rating"); plt.show()

In [ ]:
#To check normality
stat, p = normaltest(Data['num_reviews'])
print(f"Normality test: stat={stat:.3f}, p={p:.3f}")
print("Distribution:", "Normal" if p>0.05 else "Not Normal")

In [ ]:
#Testing independence between categorical features and success label
for col in ['customer_region','brand_tier','payment_method']:
    contingency = pd.crosstab(Data[col], Data['success_label'])
    stat, p, dof, expected = chi2_contingency(contingency)
    print(f"{col}: chi2={stat:.3f}, p={p:.3f}")
    print("Association:", "Strong" if p<0.05 else "Weak")

In [ ]:
#To check correlation
plt.figure(figsize=(6,4))
sns.heatmap(contingency, annot=True, fmt="d", cmap="Blues")
plt.title("Heatmap of Success Label vs Feature")
plt.ylabel("Category")
plt.xlabel("Success Label")
plt.show()

#Question-2>Develop meaningful representations of the available data.

In [ ]:
#Actual selling price
Data['discounted_price'] = Data['price'] * (1 - Data['discount_percent']/100)

In [ ]:
#Numerical features
num_features = ["price","discount_percent","avg_rating","num_reviews",
                "customer_age","past_purchase_count","discounted_price"]


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(Data,Data['success_label'], test_size=0.2, random_state=42)


In [ ]:
#Transform numerical features
scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train[num_features])
X_test_num  = scaler.transform(X_test[num_features])
X_train_num

In [ ]:
#Handle categorical attributes
X_train_cat = pd.get_dummies(X_train[['product_category','brand_tier',
                                         'customer_region','payment_method','device_used']], drop_first=True)
X_test_cat = pd.get_dummies(X_test[['product_category','brand_tier',
                                       'customer_region','payment_method','device_used']], drop_first=True)

In [ ]:
X_train_cat, X_test_cat = X_train_cat.align(X_test_cat, join='left', axis=1, fill_value=0)
X_train_cat

In [ ]:
#Extraction from textual field
vectorizer = TfidfVectorizer(stop_words='english', max_features=500)
X_train_text = vectorizer.fit_transform(X_train['review_text']).toarray()
X_test_text  = vectorizer.transform(X_test['review_text']).toarray()
X_train_text

In [ ]:
X_train = np.hstack([X_train_num, X_train_cat.values, X_train_text])
X_test  = np.hstack([X_test_num, X_test_cat.values, X_test_text])

#Question-3>Develop models to predict the outcome variable.

In [ ]:
#Prediction with structured and unstructured data as x & binary data as y
X = np.vstack([X_train,X_test])
y = Data['success_label']

In [ ]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

In [ ]:
#Baseline model approach
log_model = LogisticRegression(max_iter=1000, solver='liblinear')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(log_model, X, y, cv=cv, scoring='accuracy')
print("Logistic Regression CV Accuracy:", scores.mean())

In [ ]:
log_model = LogisticRegression(class_weight="balanced", random_state=42)
log_model.fit(X_train_res, y_train_res)
y_pred_log = log_model.predict(X_test)

In [ ]:
print("Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_log))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_log))
print("Classification Report:\n", classification_report(y_test, y_pred_log, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test, log_model.predict_proba(X_test)[:,1]))

In [ ]:
#Advanced model approach
rf_model = RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42)
scores = cross_val_score(rf_model, X, y, cv=cv, scoring='accuracy')
print("Random Forest CV Accuracy:", scores.mean())

In [ ]:
rf_model = RandomForestClassifier(class_weight="balanced", random_state=42)
rf_model.fit(X_train_res, y_train_res)
y_pred_rf = rf_model.predict(X_test)

In [ ]:
print("\nRandom Forest")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("Classification Report:\n", classification_report(y_test, y_pred_rf, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test, rf_model.predict_proba(X_test)[:,1]))

In [ ]:
#Sample prediction
def predict_outcome(price, discount_percent, avg_rating, num_reviews,
                    customer_age, past_purchase_count, product_category,
                    brand_tier, customer_region, device_used, payment_method,
                    review_text):
    discounted_price = price * (1 - discount_percent/100)
    num_input = pd.DataFrame([[price, discount_percent, avg_rating, num_reviews,
                               customer_age, past_purchase_count, discounted_price]],
                             columns=num_features)
    num_scaled = scaler.transform(num_input)
    num_sparse = csr_matrix(num_scaled)
    cat_input = pd.DataFrame([[product_category, brand_tier, customer_region,
                               payment_method, device_used]],
                             columns=['product_category','brand_tier','customer_region',
                                      'payment_method','device_used'])
    cat_encoded = pd.get_dummies(cat_input, drop_first=True)
    cat_encoded = cat_encoded.reindex(columns=X_train_cat.columns, fill_value=0)
    cat_sparse = csr_matrix(cat_encoded.values)
    text_input = vectorizer.transform([review_text])
    X_input = hstack([num_sparse, cat_sparse, text_input])
    prediction = rf_model.predict(X_input)[0]
    prob = rf_model.predict_proba(X_input)[0][1]
    return prediction, prob
pred, prob = predict_outcome(
    price=499, discount_percent=10, avg_rating=4.2, num_reviews=120,
    customer_age=28, past_purchase_count=5, product_category="Electronics",
    brand_tier="Premium", customer_region="East Zone", device_used="Mobile",
    payment_method="Credit Card", review_text="Great product, fast delivery!"
)

print("Predicted Success Label:", pred)
print("Probability of Success:", prob)
